# Mathematics — Statistics, regression and clustering
## FDP Day 5 — AI-Powered Scientific Research with Google Colab

**Objective:** Perform descriptive statistics, correlation analysis, train/test regression and K-means clustering on multivariate research data.

**Matching dataset:** `10_Mathematics_Research_Statistics_Clustering.csv`

All bundled CSVs are **synthetic teaching datasets** with plausible ranges and units, not real experimental observations. Replace them with approved research data for actual studies. Avoid uploading identifiable patient or confidential data.

**Workflow:** Upload → inspect → analyse → visualise → interpret → export.


## 1. Setup and upload the matching CSV

In [ ]:
!pip -q install seaborn scikit-learn opencv-python-headless
import io, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from IPython.display import display
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid",context="notebook")
plt.rcParams["figure.figsize"]=(9,5)
uploaded=files.upload()
csv_name=next(f for f in uploaded if f.lower().endswith(".csv"))
df=pd.read_csv(io.BytesIO(uploaded[csv_name]))
print("Loaded",csv_name,"shape:",df.shape)
display(df.head()); display(df.describe(include="all").T)


## 2. Correlation, train/test regression and clustering

In [ ]:
num=df.select_dtypes(include=np.number)
sns.heatmap(num.corr(),annot=True,cmap="vlag",center=0);plt.title("Correlation matrix");plt.show()
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_squared_error
features=["variable_A","variable_B","variable_C","variable_D"]
Xtr,Xte,ytr,yte=train_test_split(df[features],df.response_score,test_size=.25,random_state=42)
reg=LinearRegression().fit(Xtr,ytr);pred=reg.predict(Xte)
print("Test R²:",round(r2_score(yte,pred),3),"RMSE:",round(mean_squared_error(yte,pred)**.5,2))
plt.scatter(yte,pred);plt.xlabel("Observed");plt.ylabel("Predicted");plt.title("Regression: observed vs predicted");plt.show()
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
Xs=StandardScaler().fit_transform(df[features]);inertia=[]
for k in range(2,8):inertia.append(KMeans(n_clusters=k,n_init=10,random_state=42).fit(Xs).inertia_)
plt.plot(range(2,8),inertia,"o-");plt.xlabel("k");plt.ylabel("Inertia");plt.title("Elbow curve");plt.show()
labels=KMeans(n_clusters=3,n_init=10,random_state=42).fit_predict(Xs);pc=PCA(n_components=2).fit_transform(Xs)
sns.scatterplot(x=pc[:,0],y=pc[:,1],hue=labels,palette="tab10");plt.title("K-means clusters (PCA projection)");plt.show()
df["cluster"]=labels;display(df.groupby("cluster")[features+["response_score"]].mean().round(2))


## Research interpretation checklist
- State the research question, data source, sample size and units.
- Explain two findings supported by the charts.
- Identify uncertainty, confounders, measurement limitations and possible bias.
- State what additional data or validation is required before publication.
- Do not claim causation from correlation or clinical/biological diagnosis from simple image segmentation.


In [ ]:
# Export a copy of the data with any new analysis columns
df.to_csv("processed_research_data.csv",index=False)
files.download("processed_research_data.csv")
